In [1]:
#Here, we define
import numpy as np
import math
import matplotlib.pyplot as plt
from sympy import *
import sympy
from sklearn.linear_model import LinearRegression
from sympy import re, im, I, E, symbols, exp
from sklearn.model_selection import train_test_split
from scipy.optimize import minimize
import time

import tensorflow as tf

import glob

from tensorflow.keras.layers import Dense,Flatten,Cropping1D, Concatenate, MaxPooling1D, Dropout, Reshape, Conv1D, BatchNormalization, Activation, AveragePooling1D, GlobalAveragePooling1D, Lambda, Input, Concatenate, Add, UpSampling1D, Multiply
from tensorflow.keras.models import Model
from tensorflow.keras import losses
from tensorflow.keras.losses import Hinge
from tensorflow.keras import backend as K
from tensorflow.keras.losses import binary_crossentropy, categorical_crossentropy
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard, ReduceLROnPlateau,LearningRateScheduler
from tensorflow.keras.initializers import RandomNormal
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.models import Sequential
from sklearn.metrics import cohen_kappa_score, f1_score
from sklearn.model_selection import KFold, train_test_split

from scipy.special import legendre
from scipy.special import chebyt, chebyu



import numpy as np
from scipy.special import pro_ang1
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression


In [2]:
# Definimos el valor de K

def Archit(K):

# We create an arrange with K values
    Array = np.arange(1, K[0] + 1)

# e create an arrange of the form (j, K * j)
    Lay_x_Neu = np.column_stack((Array, K[1] * Array))

    return Lay_x_Neu

In [3]:
#Number of samples and parameters

max_order_slep = 35 # increase to 20 or more eventually

Nsamples = np.logspace(1, np.log10(10000), 10).astype(int)
Ntest= 1000

#Para_slep = np.linspace(1, max_order_slep+1, 10).astype(int)
Para_slep= [10]
total_runs = 20

lVal = [0,0.2, 0.4, 0.6, 0.8, 1]

In [4]:
Nsamples

array([   10,    21,    46,   100,   215,   464,  1000,  2154,  4641,
       10000])

In [5]:
# Define the samples
def samps(t):

    Values = np.random.uniform(-1, 1, int(t[0]) if isinstance(t, (list, np.ndarray)) else int(t))


    #Ordenar os valores de mayor a menor
    #OValues = np.sort(Values)  # [::-1] invierte el orden

    return Values

In [6]:
#This is the function that we approximate
def funct(x, l):
    # t is the variable
    g = np.exp(-0.5*(x-l)**2)


    return g

In [7]:
#Returns the Slepian basis function

def NorSlep(j, c):
    b = 0 # Alpha parameter for the PSAF
    theta = np.linspace(-1, 1, 200)

    psaf_values = pro_ang1(b, j, c, theta)
    psaf_values = psaf_values[0]

    psaf_values = psaf_values[1:199]
    l2_norm = np.linalg.norm(psaf_values)/np.sqrt(len(theta)-1)


    return l2_norm

norms_ = np.zeros([Para_slep[-1]])
for k in range(Para_slep[-1]):
    norms_[k] = NorSlep(k, 4)

In [8]:
#Compute the PSWF with j and c at the value x.

def PSWF(x, j, c):

    b = 0 # Alpha parameter for the PSAF

    # Calculate the PSAF values for the two modes
    psaf_values = pro_ang1(b, j, c, x)
    psaf_values = psaf_values[0]



    l2_norm = norms_[j]

    return psaf_values/l2_norm

In [9]:
#We know create the matrix for the ls approximation

def Matrix(samp, ordens):

    A= np.zeros((len(samp), ordens))
    #We create the matrix for the least square
    for j in range(ordens):
        for k in range(len(samp)):
            A[k,j]= PSWF(samp[k], j, 4)

    return A

In [ ]:
# Now we store the error for each value of l
MSE_slep = np.zeros([len(lVal), len(Para_slep), len(Nsamples), total_runs])

start_time = time.time()
total_iterations = len(lVal) * total_runs * len(Para_slep) * len(Nsamples)
counter = 0

# Loop over the different Gaussian centers l
for il, l in enumerate(lVal):

    for run in range(total_runs):

        # produce the matrix first, then take rectangles
        SamSet_ = samps(Nsamples[-1])

        # Function with the current l
        y_ = funct(SamSet_, l)

        A_ = Matrix(SamSet_, Para_slep[-1])

        TestSet = samps(Ntest)

        # Test values for this l
        Values_test = funct(TestSet, l)


        # Slepian matrix
        test_PSWF_matrix_ = np.array(
            [[PSWF(t, i, 4) for i in range(Para_slep[-1])] 
             for t in TestSet]
        )


        for orde in range(len(Para_slep)):

            for k in range(len(Nsamples)):

                # Generate sample and test sets
                A_s = A_[:Nsamples[k], :Para_slep[orde]]

                y = y_[:Nsamples[k]]


                test_PSWF_matrix = test_PSWF_matrix_[:, :Para_slep[orde]]


                # Solve least squares problem
                x = np.linalg.lstsq(A_s, y, rcond=None)[0]


                # Approximation on test set
                linear_combination_t = test_PSWF_matrix @ x


                # Error
                Err = np.linalg.norm(
                    linear_combination_t - Values_test
                ) / np.sqrt(Ntest)


                # Save error for this l
                MSE_slep[il, orde, k, run] = Err


                counter += 1
                now_time = time.time()

                print(
                    'l = ' + str(l) +
                    ', Error: ' + str(Err) +
                    ', Progress: ' + format(100*counter / total_iterations, '.2f') +
                    '%' +
                    ', approx. ' +
                    format((total_iterations-counter)*(now_time - start_time)/counter, '.1f') +
                    ' seconds remaining.'
                )# Now we store the error for each value of l
MSE_slep = np.zeros([len(lVal), len(Para_slep), len(Nsamples), total_runs])

start_time = time.time()
total_iterations = len(lVal) * total_runs * len(Para_slep) * len(Nsamples)
counter = 0

# Loop over the different Gaussian centers l
for il, l in enumerate(lVal):

    for run in range(total_runs):

        # produce the matrix first, then take rectangles
        SamSet_ = samps(Nsamples[-1])

        # Function with the current l
        y_ = funct(SamSet_, l)

        A_ = Matrix(SamSet_, Para_slep[-1])

        TestSet = samps(Ntest)

        # Test values for this l
        Values_test = funct(TestSet, l)


        # Slepian matrix
        test_PSWF_matrix_ = np.array(
            [[PSWF(t, i, 4) for i in range(Para_slep[-1])] 
             for t in TestSet]
        )


        for orde in range(len(Para_slep)):

            for k in range(len(Nsamples)):

                # Generate sample and test sets
                A_s = A_[:Nsamples[k], :Para_slep[orde]]

                y = y_[:Nsamples[k]]


                test_PSWF_matrix = test_PSWF_matrix_[:, :Para_slep[orde]]


                # Solve least squares problem
                x = np.linalg.lstsq(A_s, y, rcond=None)[0]


                # Approximation on test set
                linear_combination_t = test_PSWF_matrix @ x


                # Error
                Err = np.linalg.norm(
                    linear_combination_t - Values_test
                ) / np.sqrt(Ntest)


                # Save error for this l
                MSE_slep[il, orde, k, run] = Err


                counter += 1
                now_time = time.time()

                print(
                    'l = ' + str(l) +
                    ', Error: ' + str(Err) +
                    ', Progress: ' + format(100*counter / total_iterations, '.2f') +
                    '%' +
                    ', approx. ' +
                    format((total_iterations-counter)*(now_time - start_time)/counter, '.1f') +
                    ' seconds remaining.'
                )

l = 0, Error: 2.050190861682082e-07, Progress: 0.08%, approx. 6590.7 seconds remaining.
l = 0, Error: 1.9259688831410203e-07, Progress: 0.17%, approx. 3293.3 seconds remaining.
l = 0, Error: 1.259553700983192e-07, Progress: 0.25%, approx. 2193.8 seconds remaining.
l = 0, Error: 1.5635140619636264e-08, Progress: 0.33%, approx. 1644.0 seconds remaining.
l = 0, Error: 1.2756123607179016e-08, Progress: 0.42%, approx. 1314.2 seconds remaining.
l = 0, Error: 1.1836497359208965e-08, Progress: 0.50%, approx. 1094.3 seconds remaining.
l = 0, Error: 1.1791721968830903e-08, Progress: 0.58%, approx. 937.3 seconds remaining.
l = 0, Error: 1.1620109070638739e-08, Progress: 0.67%, approx. 819.6 seconds remaining.
l = 0, Error: 1.1561563117349798e-08, Progress: 0.75%, approx. 728.0 seconds remaining.
l = 0, Error: 1.157449811099216e-08, Progress: 0.83%, approx. 654.9 seconds remaining.
l = 0, Error: 3.3955960975267185e-07, Progress: 0.92%, approx. 1254.0 seconds remaining.
l = 0, Error: 5.902421024843

In [ ]:
# Plot Error over training set sizes
plt.figure(figsize=(12, 8))


for il, l in enumerate(lVal):

    # Average over runs
    results_mean = np.mean(MSE_slep[il], axis=2)

    # Standard deviation over runs
    results_lower = np.std(MSE_slep[il], axis=2) / results_mean


    # Since Para_slep = [10], there is only one curve
    i = 0

    plt.loglog(
        Nsamples,
        results_mean[i,:],
        label=fr"$l={l}$",
        linewidth=3
    )

    plt.fill_between(
        Nsamples,
        results_mean[i,:]/(1+results_lower[i,:]),
        results_mean[i,:]*(1+results_lower[i,:]),
        alpha=0.2
    )


plt.xlabel(r'$m$', fontsize=32, labelpad=18)
plt.ylabel(r'$\epsilon_{test}$', fontsize=32, labelpad=28)


plt.tick_params(axis='both', which='major', labelsize=28)
plt.tick_params(axis='both', which='minor', labelsize=28)


plt.legend(
    fontsize=24,
    loc='center left',
    bbox_to_anchor=(1.05, 0.5),
    ncol=1
)


plt.tight_layout()


plt.savefig("TranslatedExpo.pdf")


plt.show()

In [ ]:
results_mean_leg-results_mean_cheby

In [ ]:
results_mean - results_mean_leg

In [ ]:
results_mean - results_mean_cheby


In [ ]:
print(test_PSWF_matrix.shape)


In [ ]:
print(test_PSWF_matrix_.shape)
